In [1]:
import pandas as pd

def load_measures(file_name, system):
    qid_to_metadata = {}
    for _, i in pd.read_csv('../test-2025/test-2025-mapping.tsv', sep="\t").iterrows():
        qid_to_metadata[str(i["query_id"])] = {k:i[k] for k in ["source", "llm", "domain"]}
    ret = []
    for _, i in pd.read_csv(file_name, sep="\t", names=["measure", "qid", "value"]).iterrows():
        i = {k: v.strip() for k, v in i.to_dict().items()}
        if i["measure"] == 'relstring':
            continue
        if i["qid"] not in qid_to_metadata:
            continue
        i["value"] = float(i["value"])
        i["system"] = system
        
        i.update(qid_to_metadata[i["qid"]])
        ret.append(i)
    return pd.DataFrame(ret)

In [2]:
df_eval = pd.concat([
    load_measures('../pyterrier-bm25-retrieval/runs/bm25/test/trec_eval.tsv', 'BM25 (Pyterrier)'),
    load_measures('../anserini-bm25-retrieval/runs/test/trec_eval.tsv', 'BM25 (Anserini)'),
    load_measures('../lightning-dense-retrieval/runs/test/tot-2025-index/trec_eval.tsv', 'Dense Retrieval')
])
    

In [68]:
df_eval

,measure,qid,value,system,source,llm,domain
0,num_ret,3000,1000.0,BM25 (Pyterrier),synthetic,gpt-4o,cheese
1,num_rel,3000,1.0,BM25 (Pyterrier),synthetic,gpt-4o,cheese
2,num_rel_ret,3000,0.0,BM25 (Pyterrier),synthetic,gpt-4o,cheese
3,map,3000,0.0,BM25 (Pyterrier),synthetic,gpt-4o,cheese
4,Rprec,3000,0.0,BM25 (Pyterrier),synthetic,gpt-4o,cheese
...,...,...,...,...,...,...,...
55975,set_relative_P,3621,0.0,Dense Retrieval,synthetic,Llama-3.1-8B-Instruct,encryption method
55976,set_recall,3621,0.0,Dense Retrieval,synthetic,Llama-3.1-8B-Instruct,encryption method
55977,set_map,3621,0.0,Dense Retrieval,synthetic,Llama-3.1-8B-Instruct,encryption method
55978,set_F,3621,0.0,Dense Retrieval,synthetic,Llama-3.1-8B-Instruct,encryption method


In [74]:
df_eval_systems = df_eval[["measure", "value", "system"]].groupby(["measure", "system"]).aggregate({"value": "mean"}).reset_index()[["system", "measure", "value"]]

df_eval_systems[df_eval_systems["measure"].isin(["recall_1000"])]

,system,measure,value
192,BM25 (Anserini),recall_1000,0.276699
193,BM25 (Pyterrier),recall_1000,0.485531
194,Dense Retrieval,recall_1000,0.265273


In [ ]:
df_eval_source = df_eval[["measure", "value", "system", "source"]].groupby(["measure", "system", "source"]).aggregate({"value": "mean"}).reset_index()[["source", "system", "measure", "value"]].sort_values(["source", "system"])

df_eval_source[df_eval_source["measure"].isin(["recall_1000"])]

,source,system,measure,value
576,ms-tot,BM25 (Anserini),recall_1000,0.209302
579,ms-tot,BM25 (Pyterrier),recall_1000,0.430233
582,ms-tot,Dense Retrieval,recall_1000,0.156977
577,nist,BM25 (Anserini),recall_1000,0.738255
580,nist,BM25 (Pyterrier),recall_1000,0.706667
583,nist,Dense Retrieval,recall_1000,0.466667
578,synthetic,BM25 (Anserini),recall_1000,0.084175
581,synthetic,BM25 (Pyterrier),recall_1000,0.406667
584,synthetic,Dense Retrieval,recall_1000,0.226667


In [4]:
DOMAIN_GROUPS = {
    'aircraft occurrence': 'science',
    'artificial fly': 'science',
    'algorithm': 'science',
    'astronomical event': 'science',
    'cpu architecture': 'science',
    'cloud type': 'science',
    'encryption method': 'science',
    'mathematical statement': 'science',
    'enzyme': 'science',
    'software': 'science',
    'gene': 'science',
    'laboratory equipment': 'science',
    'medical condition': 'science',
    'molecular geometry': 'science',
    'networking protocol': 'science',
    'online service': 'science',
    'probability distribution': 'science',
    'pseudoscience': 'science',

    'asian comic series': 'comic',

    'attraction': 'attraction',

    'australian political party': 'politics',
    'referendum': 'politics',

    'buddhist term': 'religion',
    'religious building': 'religion',

    'celebrity': 'person',
    'scientist': 'person',
    'economist': 'person',
    'japanese clan': 'person',
    'tribe': 'person',
    'yacht club': 'person',
    
    'landmark': 'landmark',

    'cheese': 'food',
    'cocktail': 'food',

    'chess opening': 'games',

    'clothing type': 'sports',
    
    'movie': 'movie',
    'movies': 'movie',

    'court': 'law',
    'court case': 'law',
    'u.s. legislation': 'law',
    'eu legislation': 'law',
    'indian constituency': 'law',

    'diving equipment': 'sports',
    'football match': 'sports',
    'judo technique': 'sports',
    'pinball': 'sports',

    'film awards': 'event',
    'public transit accident': 'event', 
    'summit meeting': 'event',

    'financial index': 'finance',
  
    'magazine': 'book',

    'musical interval': 'music',
    'musical scale': 'music',
    'radio show': 'music',
}

In [6]:
df_eval_domain["domain"].unique()

array(['attraction', 'book', 'comic', 'event', 'finance', 'food', 'games',
       'landmark', 'law', 'movie', 'music', 'person', 'politics',
       'religion', 'science', 'sports'], dtype=object)

In [5]:
df_eval_domain = df_eval[["measure", "value", "system", "domain"]].copy()
df_eval_domain["domain"] = df_eval_domain["domain"].apply(lambda i: DOMAIN_GROUPS[i])

df_eval_domain = df_eval_domain.groupby(["measure", "system", "domain"]).aggregate({"value": "mean"}).reset_index()[["domain", "system", "measure", "value"]].sort_values(["domain", "system"])

df_eval_domain[df_eval_domain["measure"].isin(["recall_1000"])]

,domain,system,measure,value
3072,attraction,BM25 (Anserini),recall_1000,0.000000
3088,attraction,BM25 (Pyterrier),recall_1000,0.500000
3104,attraction,Dense Retrieval,recall_1000,0.166667
3073,book,BM25 (Anserini),recall_1000,0.000000
3089,book,BM25 (Pyterrier),recall_1000,0.000000
3105,book,Dense Retrieval,recall_1000,0.333333
3074,comic,BM25 (Anserini),recall_1000,0.166667
3090,comic,BM25 (Pyterrier),recall_1000,0.500000
3106,comic,Dense Retrieval,recall_1000,0.000000
3075,event,BM25 (Anserini),recall_1000,0.055556
